## 🎯 Learning Objectives
* Understand the purpose and process of merging LoRA adapters into a base LLM.
* Grasp the concept of model quantisation and its benefits for deployment.
* Learn how to apply different quantisation techniques to a merged LLM.
* Evaluate the trade-offs between model size, inference speed, and performance when merging and quantising LLMs.


## Merging LoRA Adapters and Quantisation: Streamlining LLM Deployment

In the previous lessons, we mastered finetuning Large Language Models (LLMs) using Low-Rank Adaptation (LoRA). LoRA allows us to adapt a massive base model to a specific task or dataset by training only a small set of adapter weights, significantly reducing computational cost and memory footprint during training. However, when it comes to **deployment**, these separate adapter weights can introduce complexities.

### The Need for Merging LoRA Adapters

Imagine you've trained a LoRA adapter that makes a general-purpose LLM an expert in medical diagnostics. For inference, you currently need to load the large base model *and* the smaller LoRA adapter. This means:

1.  **Increased Memory Footprint**: Both the base model and the adapter weights reside in memory, even if the adapter is small.
2.  **Deployment Complexity**: Managing two separate sets of weights (base + adapter) can be cumbersome, especially across different environments or when serving multiple models.
3.  **Potential Inference Overhead**: While minimal, there's a slight overhead in applying the adapter weights during each forward pass.

**Merging LoRA adapters** solves these issues by integrating the adapter weights directly into the base model's weights. Conceptually, if the base model is a large painting and the LoRA adapter is a transparent overlay that subtly changes certain colors, merging is like permanently painting those changes onto the original canvas. The result is a single, unified model that behaves exactly like the base model with the LoRA adapter applied, but without the need for separate files or runtime composition.

This merged model is then a standalone entity, ready for simpler deployment and potentially faster inference, as the adapter's computations are now baked directly into the model's primary weight matrices.

### The Power of Quantisation

Even after merging, LLMs remain notoriously large. A 7-billion parameter model, stored in `float16` (half-precision), occupies approximately 14 GB of memory. This is a significant barrier for deployment on consumer-grade GPUs, edge devices, or even cost-effective cloud instances.

**Quantisation** is a technique that reduces the precision of a model's weights (and sometimes activations) from higher-precision formats (like `float32` or `float16`) to lower-precision formats (like `int8`, `int4`, or even `int2`). Think of it like compressing a high-resolution image into a smaller file size – you lose some detail, but the overall picture remains recognizable and much easier to store and transmit.

Key benefits of quantisation:

1.  **Reduced Memory Footprint**: Drastically shrinks the model size, allowing larger models to fit into limited GPU memory or even run on CPUs.
2.  **Faster Inference**: Lower precision operations can be executed much faster by modern hardware, leading to significant speedups.
3.  **Lower Energy Consumption**: Less data movement and simpler computations translate to reduced power usage.

However, quantisation is a trade-off: reducing precision can lead to a slight degradation in model performance (e.g., perplexity, accuracy). The goal is to find the sweet spot where memory and speed gains outweigh the minimal performance loss.

Modern quantisation techniques, such as **BitsAndBytes (BNB)** 4-bit quantisation, **GPTQ**, and **AWQ**, are highly optimized to minimize this performance drop. They often involve sophisticated calibration steps to determine the best scaling factors for converting weights to lower precision. For instance, BNB's `NF4` (NormalFloat 4-bit) quantisation is specifically designed for weights that follow a normal distribution, common in neural networks.

In practice, you'll often merge your LoRA adapters first to create a unified model, and then quantise this merged model for efficient deployment. This two-step process ensures you get the benefits of both adaptation and compression.


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install transformers peft accelerate bitsandbytes torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model
import os

# --- Configuration --- 
# Using a small model for demonstration purposes to keep download times reasonable.
# For real-world applications, you'd use a larger base model.
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Define a dummy LoRA configuration for demonstration.
# In a real scenario, you would load an actual trained adapter.
LORA_CONFIG = LoraConfig(
    r=8, # LoRA attention dimension
    lora_alpha=16, # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "v_proj"], # Modules to apply LoRA to
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Bias type for LoRA layers
    task_type="CAUSAL_LM" # Task type
)

# --- Step 1: Load Base Model --- 
print(f"\n--- Loading base model: {BASE_MODEL_ID} ---")
# Load the base model in float16 precision
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

# Get initial memory usage of the base model
if torch.cuda.is_available():
    initial_base_memory_mb = torch.cuda.memory_allocated() / (1024**2)
    print(f"Initial base model memory usage (float16): {initial_base_memory_mb:.2f} MB")
else:
    print("CUDA not available. Memory usage will not be reported for GPU.")

# --- Step 2: Simulate LoRA Adapter Loading (or create a dummy one) ---
# In a real scenario, you would load a trained adapter like this:
# peft_model = PeftModel.from_pretrained(base_model, "path/to/your/lora_adapter")

# For this example, we'll create a dummy PeftModel by applying a new LoRA config
# This simulates having a LoRA adapter ready to be merged.
print("\n--- Applying dummy LoRA adapter to base model ---")
peft_model = get_peft_model(base_model, LORA_CONFIG)
peft_model.print_trainable_parameters()

# --- Step 3: Merge LoRA Adapters into the Base Model ---
print("\n--- Merging LoRA adapters into the base model ---")
# The `merge_and_unload()` method merges the LoRA weights into the base model's weights
# and returns a new `AutoModelForCausalLM` instance with the merged weights.
merged_model = peft_model.merge_and_unload()

# The `merged_model` is now a standard `AutoModelForCausalLM` instance
# without any PEFT layers. It's in the original precision (float16 in this case).

if torch.cuda.is_available():
    merged_model_memory_mb = torch.cuda.memory_allocated() / (1024**2)
    print(f"Merged model memory usage (float16): {merged_model_memory_mb:.2f} MB")
    # Note: The memory might appear similar to the base model if the base model was already loaded
    # and the PEFT model was just a wrapper. The key is that `merged_model` is now standalone.
else:
    print("CUDA not available. Memory usage will not be reported for GPU.")

print("Merged model type:", type(merged_model))
print("Example inference with merged model:")
input_text = "The capital of France is"
inputs = tokenizer(input_text, return_tensors="pt").to(merged_model.device)
with torch.no_grad():
    outputs = merged_model.generate(**inputs, max_new_tokens=10)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# --- Step 4: Quantise the Merged Model (e.g., to 4-bit) ---
print("\n--- Quantising the merged model to 4-bit using BitsAndBytes ---")

# Define 4-bit quantisation configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, # Enable 4-bit quantisation
    bnb_4bit_quant_type="nf4", # Use NF4 quantisation (recommended for weights)
    bnb_4bit_compute_dtype=torch.bfloat16, # Compute in bfloat16 for better precision during operations
    bnb_4bit_use_double_quant=True, # Double quantisation for even smaller memory footprint
)

# Reload the base model directly with quantisation config
# This is the most straightforward way to load a model in 4-bit.
# If you had saved the merged model, you would load it from disk with this config.
quantised_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, # We're reloading the base model, but applying quantisation directly
    # In a real scenario, you'd save `merged_model` and then load it with quantisation.
    # For this demo, we'll just show the effect on the base model for simplicity.
    quantization_config=quantization_config,
    device_map="auto"
)

# If you had saved the merged model to disk, you would load it like this:
# merged_model.save_pretrained("./merged_model_output")
# quantised_model = AutoModelForCausalLM.from_pretrained(
#     "./merged_model_output",
#     quantization_config=quantization_config,
#     device_map="auto"
# )

if torch.cuda.is_available():
    quantised_model_memory_mb = torch.cuda.memory_allocated() / (1024**2)
    print(f"Quantised model memory usage (4-bit): {quantised_model_memory_mb:.2f} MB")
    print(f"Memory reduction from float16 base to 4-bit quantised: "
          f"{(initial_base_memory_mb - quantised_model_memory_mb) / initial_base_memory_mb * 100:.2f}%")
else:
    print("CUDA not available. Memory usage will not be reported for GPU.")

print("Quantised model type:", type(quantised_model))
print("Example inference with quantised model:")
inputs = tokenizer(input_text, return_tensors="pt").to(quantised_model.device)
with torch.no_grad():
    outputs = quantised_model.generate(**inputs, max_new_tokens=10)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Clean up (optional)
if torch.cuda.is_available():
    del base_model, peft_model, merged_model, quantised_model
    torch.cuda.empty_cache()
    print("\nCUDA memory cleared.")


### Interpreting the Code Output and Understanding Trade-offs

The code demonstrates a typical workflow for preparing a LoRA-finetuned LLM for deployment: merging the adapters and then quantising the resulting model.

1.  **Initial Base Model Loading**: We first load the `TinyLlama-1.1B-Chat-v1.0` model in `float16` precision. The reported memory usage (if CUDA is available) gives us a baseline for comparison. For a 1.1B parameter model, `float16` means roughly 2.2 GB (1.1B * 2 bytes/param).

2.  **Simulating LoRA Adapter**: We use `get_peft_model` to wrap our base model with a `LoraConfig`. While we don't load pre-trained weights here, this step conceptually represents having a LoRA adapter applied. The `print_trainable_parameters()` output shows the tiny fraction of parameters that LoRA modifies.

3.  **Merging LoRA Adapters**: The `peft_model.merge_and_unload()` call is crucial. It takes the LoRA weights and mathematically combines them with the base model's weights. The returned `merged_model` is a standard `AutoModelForCausalLM` instance, no longer a `PeftModel`. This means it's a single, self-contained model. The memory footprint of the `merged_model` will be very similar to the original `base_model` because the LoRA weights are small and are integrated, not added as a separate large block. The key benefit here is **simplicity of deployment** and **removal of PEFT runtime overhead**.

4.  **Quantising the Merged Model**: We then demonstrate 4-bit quantisation using `BitsAndBytesConfig`. When `load_in_4bit=True` is passed to `AutoModelForCausalLM.from_pretrained()`, the model's weights are loaded directly into 4-bit precision. You'll observe a dramatic reduction in memory usage – typically around 75% compared to the `float16` version. For our 1.1B model, this means going from ~2.2 GB to ~0.55 GB. This is a game-changer for fitting models on smaller GPUs or even running multiple models concurrently.

    *   `bnb_4bit_quant_type="nf4"`: Specifies the NormalFloat 4-bit quantisation scheme, which is optimized for neural network weights.
    *   `bnb_4bit_compute_dtype=torch.bfloat16`: During computation, the 4-bit weights are de-quantised to `bfloat16` for better numerical stability than `float16`.
    *   `bnb_4bit_use_double_quant=True`: Applies a second quantisation step to the quantisation constants, further reducing memory slightly.

### Performance Trade-offs and Use Cases

| Feature        | Merging LoRA Adapters                                  | Quantisation (e.g., 4-bit)                               | Combined (Merged + Quantised)                                  |
| :------------- | :----------------------------------------------------- | :------------------------------------------------------- | :------------------------------------------------------------- |
| **Memory**     | Similar to base model (LoRA weights are small)         | Significantly reduced (e.g., 75% for 4-bit)              | Significantly reduced                                          |
| **Inference**  | Potentially slightly faster (no PEFT overhead)         | Significantly faster (hardware-dependent)                | Significantly faster, highly efficient                         |
| **Complexity** | Simplifies deployment (single model file)              | Adds a layer of complexity (calibration, potential quality loss) | Simplified deployment, but with quantisation considerations    |
| **Quality**    | Identical to base + LoRA (no information loss)         | Minor to moderate degradation (depends on method & model) | Minor to moderate degradation                                  |
| **Portability**| Highly portable (standard `transformers` model)        | Highly portable (standard `transformers` model, but requires `bitsandbytes` or similar) | Highly portable                                                |

**When to Merge?**

*   **Production Deployment**: When you need a single, self-contained model artifact for easier versioning, distribution, and serving.
*   **Inference Optimization**: To eliminate the minor runtime overhead of applying LoRA adapters.
*   **Saving to Disk**: If you want to save the finetuned model as a single checkpoint.

**When to Quantise?**

*   **Resource-Constrained Environments**: Edge devices, mobile phones, consumer GPUs (e.g., 8GB VRAM cards).
*   **Cost-Sensitive Cloud Deployments**: To reduce GPU instance costs by using smaller GPUs or fitting more models per GPU.
*   **High-Throughput Inference**: To achieve faster inference speeds, especially when batching requests.
*   **Large Models**: Essential for deploying models like Llama-70B or Mixtral-8x7B, which are otherwise too large for most single GPUs.

**Modern Quantisation Methods (2026 Context)**:

*   **BitsAndBytes (BNB)**: Widely used for 8-bit and 4-bit quantisation, especially `NF4`. Excellent balance of ease of use, memory reduction, and performance. Often used for loading models directly.
*   **GPTQ**: A post-training quantisation (PTQ) method that quantises weights to 4-bit or 3-bit with minimal perplexity loss. Requires a small calibration dataset. Libraries like `auto-gptq` provide easy integration.
*   **AWQ (Activation-aware Weight Quantisation)**: Another PTQ method that focuses on quantising weights based on activation magnitudes, aiming for even better performance preservation than GPTQ. Libraries like `autoawq` are available.
*   **KV-Cache Quantisation**: Beyond just weights, quantising the Key-Value cache during inference can further reduce memory usage and improve throughput, especially for long contexts. This is an active area of research and implementation.

By combining LoRA merging with advanced quantisation techniques, ML engineers can deploy highly specialized LLMs efficiently and cost-effectively, bringing powerful AI capabilities to a wider range of applications and hardware.


### Resources

*   **Hugging Face PEFT Library Documentation**: [https://huggingface.co/docs/peft/index](https://huggingface.co/docs/peft/index)
*   **Hugging Face `merge_and_unload` Example**: [https://huggingface.co/docs/peft/v0.7.0/en/package_reference/peft_model#peft.PeftModel.merge_and_unload](https://huggingface.co/docs/peft/v0.7.0/en/package_reference/peft_model#peft.PeftModel.merge_and_unload)
*   **Hugging Face Quantisation (BitsAndBytes) Integration**: [https://huggingface.co/docs/transformers/main/en/quantization](https://huggingface.co/docs/transformers/main/en/quantization)
*   **BitsAndBytes GitHub Repository**: [https://github.com/TimDettmers/bitsandbytes](https://github.com/TimDettmers/bitsandbytes)
*   **GPTQ Paper**: [https://arxiv.org/abs/2210.17323](https://arxiv.org/abs/2210.17323)
*   **AWQ Paper**: [https://arxiv.org/abs/2306.00978](https://arxiv.org/abs/2306.00978)
*   **PyTorch Quantisation Documentation**: [https://pytorch.org/docs/stable/quantization.html](https://pytorch.org/docs/stable/quantization.html)
